# 🔤 Brahmi Script OCR — TrOCR Training on Google Colab

This notebook trains TrOCR on **combined Brahmi datasets** (synthetic + Capstone handwritten + BrahmiGAN + Kaggle) using Colab's **free GPU**.

**Pipeline:**
1. Mount Google Drive & clone project
2. Install dependencies
3. Load/Generate synthetic dataset (~20K images, cached to Drive to save GPU time!)
4. Upload Extra datasets from Drive (Capstone, BrahmiGAN, Kaggle Archive)
   * **TIP:** Uploading ZIP files is much faster than uploading folders!
5. Fine-tune `microsoft/trocr-small-printed` on combined ~107K images
6. Test inference
7. Save model to Drive

---

> ⚠️ **Before running:** Go to **Runtime → Change runtime type → GPU (T4)**

## Step 0 — Verify GPU is available

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Clone project from GitHub

In [ ]:
import os
import shutil

PROJECT_DIR = '/content/brahmi_ocr_project'

# Clone fresh from GitHub
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

!git clone https://github.com/tejaskarade100/brahmi_ocr_project.git {PROJECT_DIR}

%cd {PROJECT_DIR}
!ls

## Step 3 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

!pip install -q editdistance


## Step 4a — Load/Generate synthetic Brahmi dataset

Generates **20,000 images** (5k chars + 10k words + 5k phrases) with augmentations.

**GPU TIME SAVER:** If previously generated, it unzips directly from Google Drive!
Takes ~5–10 minutes only on the first run.

In [ ]:
DRIVE_SYNTHETIC_ZIP = '/content/drive/MyDrive/Project/brahmi_ocr_project/synthetic_dataset.zip'
LOCAL_ZIP = '/content/brahmi_ocr_project/synthetic_dataset.zip'
DATASET_DIR = '/content/brahmi_ocr_project/dataset'

if os.path.exists(f'{DATASET_DIR}/splits/train.txt'):
    print("✅ Synthetic dataset is already present locally. Skipping generation.")
elif os.path.exists(DRIVE_SYNTHETIC_ZIP):
    print(f"📦 Found cached synthetic dataset ZIP on Drive!")
    print(f"Unzipping {DRIVE_SYNTHETIC_ZIP} (this is much faster than generating)...")
    os.makedirs(os.path.dirname(LOCAL_ZIP), exist_ok=True)
    shutil.copy2(DRIVE_SYNTHETIC_ZIP, LOCAL_ZIP)
    !unzip -q -o {LOCAL_ZIP} -d /content/brahmi_ocr_project/
    print("✅ Synthetic dataset loaded successfully from Drive.")
else:
    print("⏳ Cached synthetic dataset not found on Drive.")
    print("Generating from scratch (takes ~15-25 mins)...")
    !python dataset/generate_synthetic.py --num_chars 40000 --num_words 10000 --num_phrases 40000
    
    print("\n🗜 Zipping dataset for future use to save Colab GPU time...")
    !zip -q -r {LOCAL_ZIP} dataset/images dataset/labels.txt dataset/splits
    
    print(f"💾 Saving to Google Drive at {DRIVE_SYNTHETIC_ZIP}...")
    os.makedirs(os.path.dirname(DRIVE_SYNTHETIC_ZIP), exist_ok=True)
    shutil.copy2(LOCAL_ZIP, DRIVE_SYNTHETIC_ZIP)
    print("✅ Saved to Drive! Next time you run this notebook, generation will be skipped.")

In [ ]:
import os
# Verify dataset integrity
if os.path.exists('dataset/images'):
    print(f"Total images: {len(os.listdir('dataset/images')) - 1}")  # minus .gitkeep
else:
    print("dataset/images not found!")
for split in ['train', 'val', 'test']:
    split_file = f'dataset/splits/{split}.txt'
    if os.path.exists(split_file):
        lines = sum(1 for line in open(split_file) if line.strip())
        print(f"{split.capitalize()} split : {lines}")
    else:
        print(f"⚠️ {split_file} not found! You might need to regenerate the dataset or run a split script.")

## Step 4b — Upload Capstone handwritten dataset from Drive

**TIP:** Upload `capstone_data.zip` to Drive for much faster transfer!
This cell will look for the ZIP first, then the folder.

In [ ]:
# Try ZIP paths first
CAPSTONE_ZIP_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/capstone_data.zip',
    '/content/drive/MyDrive/capstone_data.zip',
]

CAPSTONE_FOLDER_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/Capstone_Brahmi_Inscriptions/OCR/OCR_Dataset/RecognizerDataset_150_210',
    '/content/drive/MyDrive/Capstone_Brahmi_Inscriptions/OCR/OCR_Dataset/RecognizerDataset_150_210',
    '/content/drive/MyDrive/RecognizerDataset_150_210',
]

CAPSTONE_LOCAL = '/content/brahmi_ocr_project/capstone_data'
CAPSTONE_FOUND = False

if not os.path.exists(CAPSTONE_LOCAL):
    # 1. Check for ZIP
    for zip_path in CAPSTONE_ZIP_CANDIDATES:
        if os.path.exists(zip_path):
            print(f"📦 Found Capstone ZIP: {zip_path}")
            !unzip -q -o {zip_path} -d {CAPSTONE_LOCAL}
            CAPSTONE_FOUND = True
            break
    
    # 2. Check for Folder (fallback)
    if not CAPSTONE_FOUND:
        for path in CAPSTONE_FOLDER_CANDIDATES:
            if os.path.exists(path):
                print(f'📁 Copying Capstone Folder: {path}...')
                shutil.copytree(path, CAPSTONE_LOCAL)
                CAPSTONE_FOUND = True
                break
    CAPSTONE_FOUND = True

if CAPSTONE_FOUND:
    num_folders = len([d for d in os.listdir(CAPSTONE_LOCAL) if os.path.isdir(os.path.join(CAPSTONE_LOCAL, d))])
    print(f'✅ Capstone dataset ready: {num_folders} character folders')
    print('⚠️ Capstone dataset not found on Drive.')

## Step 4c — Upload BrahmiGAN dataset from Drive

**TIP:** Upload `brahmigan_data.zip` to Drive for much faster transfer!

In [ ]:
# Try ZIP paths first
BRAHMIGAN_ZIP_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/brahmigan_data.zip',
    '/content/drive/MyDrive/brahmigan_data.zip',
]
BRAHMIGAN_FOLDER_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/BrahmiGAN/BrahmiGAN',
    '/content/drive/MyDrive/BrahmiGAN/BrahmiGAN',
    '/content/drive/MyDrive/BrahmiGAN',
]

BRAHMIGAN_LOCAL = '/content/brahmi_ocr_project/brahmigan_data'
BRAHMIGAN_FOUND = False

if not os.path.exists(BRAHMIGAN_LOCAL):
    # 1. Check for ZIP
    for zip_path in BRAHMIGAN_ZIP_CANDIDATES:
        if os.path.exists(zip_path):
            print(f"📦 Found BrahmiGAN ZIP: {zip_path}")
            !unzip -q -o {zip_path} -d {BRAHMIGAN_LOCAL}
            BRAHMIGAN_FOUND = True
            break
    
    # 2. Check for Folder
    if not BRAHMIGAN_FOUND:
        for path in BRAHMIGAN_FOLDER_CANDIDATES:
            if os.path.exists(path):
                print(f'📁 Copying BrahmiGAN Folder: {path}...')
                shutil.copytree(path, BRAHMIGAN_LOCAL)
                BRAHMIGAN_FOUND = True
                break
    BRAHMIGAN_FOUND = True

if BRAHMIGAN_FOUND:
    num_folders = len([d for d in os.listdir(BRAHMIGAN_LOCAL) if os.path.isdir(os.path.join(BRAHMIGAN_LOCAL, d))])
    print(f'✅ BrahmiGAN dataset ready: {num_folders} character folders')
    print('⚠️ BrahmiGAN dataset not found on Drive.')

## Step 4d — Upload Kaggle Archive dataset from Drive

**TIP:** Upload `kaggle_data.zip` to Drive for much faster transfer!

In [ ]:
# Try ZIP paths first
KAGGLE_ZIP_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/kaggle_data.zip',
    '/content/drive/MyDrive/kaggle_data.zip',
]
KAGGLE_FOLDER_CANDIDATES = [
    '/content/drive/MyDrive/Project/brahmi_ocr_project/archive/train',
    '/content/drive/MyDrive/archive/train',
]

KAGGLE_LOCAL = '/content/brahmi_ocr_project/kaggle_data'
KAGGLE_FOUND = False

if not os.path.exists(KAGGLE_LOCAL):
    # 1. Check for ZIP
    for zip_path in KAGGLE_ZIP_CANDIDATES:
        if os.path.exists(zip_path):
            print(f"📦 Found Kaggle ZIP: {zip_path}")
            !unzip -q -o {zip_path} -d {KAGGLE_LOCAL}
            KAGGLE_FOUND = True
            break
    
    # 2. Check for Folder
    if not KAGGLE_FOUND:
        for path in KAGGLE_FOLDER_CANDIDATES:
            if os.path.exists(path):
                print(f'📁 Copying Kaggle Folder: {path}...')
                shutil.copytree(path, KAGGLE_LOCAL)
                KAGGLE_FOUND = True
                break

# Kaggle is intentionally excluded from training due uncertain label mapping.
# if KAGGLE_FOUND:
#     num_folders = len([d for d in os.listdir(KAGGLE_LOCAL) if os.path.isdir(os.path.join(KAGGLE_LOCAL, d))])
#     print(f'✅ Kaggle dataset ready: {num_folders} character folders')
# else:
#     print('⚠️ Kaggle dataset not found on Drive. Skipping.')

## Step 5 — Train the TrOCR model 🚀

| Setting | Value |
|---------|-------|
| Base model | `microsoft/trocr-small-printed` |
| Epochs | 15 |
| Batch size | 8 |
| Learning rate | 3e-5 |
| FP16 | Auto (enabled on GPU) |
| Data | Synthetic (~90K) + Capstone (~60K) + BrahmiGAN (~21K) |

**Estimated time:** ~60–120 min on T4 GPU

In [ ]:
# Build training command
EXTRA_DATA_PATHS = []
if CAPSTONE_FOUND:
    EXTRA_DATA_PATHS.append(CAPSTONE_LOCAL)
if BRAHMIGAN_FOUND:
    EXTRA_DATA_PATHS.append(BRAHMIGAN_LOCAL)
# if KAGGLE_FOUND:
#     EXTRA_DATA_PATHS.append(KAGGLE_LOCAL)

EXTRA_DATA_FLAG = ''
if len(EXTRA_DATA_PATHS) > 0:
    EXTRA_DATA_FLAG = f'--extra_data ' + ' '.join(EXTRA_DATA_PATHS)
    print(f'✅ Training with COMBINED data (synthetic + {len(EXTRA_DATA_PATHS)} extra datasets)')
else:
    print(f'ℹ️ Training with SYNTHETIC data only')

!python training/train.py \
    --epochs 15 \
    --batch_size 8 \
    --lr 3e-5 \
    --data_dir dataset \
    --output_dir model/brahmi_trocr \
    --drive_save_path /content/drive/MyDrive/Project/brahmi_ocr_project/model/brahmi_trocr \
    --resume_checkpoint /content/drive/MyDrive/Project/brahmi_ocr_project/model/last_checkpoint.pt \
    --resume \
    {EXTRA_DATA_FLAG}

## Step 6 — Test inference on sample images

In [ ]:
# Load labels for ground truth comparison
import sys
sys.path.insert(0, PROJECT_DIR)

labels = {}
labels_path = 'dataset/labels.txt'
if os.path.exists(labels_path):
    for line in open(labels_path, encoding='utf-8'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            labels[parts[0]] = parts[1]

# Run inference on first test image
test_split_path = 'dataset/splits/test.txt'
if os.path.exists(test_split_path):
    test_images = [line.strip() for line in open(test_split_path) if line.strip()]
    if test_images:
        test_img = test_images[0]
        print(f"Test image   : {test_img}")
        print(f"Ground truth : {labels.get(test_img, 'N/A')}")
        print()
        !python inference/predict.py --image dataset/images/{test_img} --model_dir model/brahmi_trocr

In [ ]:
# Batch test: compare predictions vs ground truth
from inference.predict import load_trained_model, predict

if os.path.exists('model/brahmi_trocr'):
    processor, model, device = load_trained_model('model/brahmi_trocr')
    print(f"Model loaded on: {device}\n")

    correct = 0
    test_split_path = 'dataset/splits/test.txt'
    if os.path.exists(test_split_path):
        test_images = [line.strip() for line in open(test_split_path) if line.strip()]
        total = min(10, len(test_images))
        for img_name in test_images[:total]:
            img_path = f'dataset/images/{img_name}'
            pred = predict(img_path, processor, model, device)
            gt = labels.get(img_name, '')
            match = '✅' if pred.strip() == gt.strip() else '❌'
            if pred.strip() == gt.strip():
                correct += 1
            print(f"{match} {img_name}  |  GT: {gt}  |  Pred: {pred}")

        print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.0f}%)")

## Step 7 — Save trained model to Google Drive

Copies the model to your Drive at: `My Drive/Project/brahmi_ocr_project/model/brahmi_trocr/`

In [ ]:
import shutil

# Save model to your Google Drive project folder
DRIVE_MODEL_DIR = '/content/drive/MyDrive/Project/brahmi_ocr_project/model/brahmi_trocr'

os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)

if os.path.exists(DRIVE_MODEL_DIR):
    shutil.rmtree(DRIVE_MODEL_DIR)

shutil.copytree('model/brahmi_trocr', DRIVE_MODEL_DIR)
print(f"✅ Model saved to Google Drive!")
print(f"   Path: {DRIVE_MODEL_DIR}")
print(f"   Files: {os.listdir(DRIVE_MODEL_DIR)}")

## Done! 🎉

### Your trained model is saved at:
📁 `Google Drive / Project / brahmi_ocr_project / model / brahmi_trocr /`

### To use locally:
1. Download the `brahmi_trocr` folder from Drive to your local `model/` folder
2. Run inference:
   ```bash
   python inference/predict.py --image <your_image.png> --model_dir model/brahmi_trocr
   ```